In [1]:
import os
from dotenv import load_dotenv

In [2]:
load_dotenv() 

#Assert is similar to if condition just if the condition is false then a error Message is displayed i.e Set the GROQ_API_KEY in a .env file or above
assert os.getenv("GROQ_API_KEY"), "Set the GROQ_API_KEY in a .env file or above"


print("API key loaded.")

API key loaded.


---
**RAG solution**
-
Retrieve — Find the most relevant passages from your documents for the user's question

Generate — Feed those passages as context to the LLM so it can answer accurately
```
┌─────────────┐     embed      ┌──────────────────┐
│  Your PDF   │ ─────────────► │  Vector Store    │
│  (chunks)   │                │  (Chroma DB)     │
└─────────────┘                └────────┬─────────┘
                                        │  similarity search
┌─────────────┐     embed               ▼
│  Question   │ ─────────────► top-k relevant chunks
└─────────────┘                         │
                                        ▼
                             ┌──────────────────────┐
                             │  LLM (Qwen3 / Groq)  │
                             │  context + question  │
                             └──────────┬───────────┘
                                        │
                                        ▼
                                     Answer
```                                    

In [3]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"  # As the notebook and the pdf are in same folder

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the PDF.")
print("\n--- First page preview (first 500 chars) ---")
print(pages[0].page_content[:500])

D:\Temp\ipykernel_7044\1199836156.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 9 pages from the PDF.

--- First page preview (first 500 chars) ---
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [4]:
print("----Second Page-----")
print(pages[1].page_content[:500])

----Second Page-----
Telecom Technical Reference Guide  - Internal Use Only
1. Introduction to Mobile Networks
Mobile networks have evolved through several generations, each offering significant improvements in speed,
capacity, and capability.
2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to
around 50 kbps, sufficient only for text messaging and simple email.
3G (UMTS/HSPA) networks brought mobile broadband, enabling video calls, mobile internet browsing, an


---
**Key parameters:**
-
- `chunk_size` — max characters per chunk
- `chunk_overlap` — characters shared between neighbouring chunks (prevents losing context at boundaries)

```
PDF Document
     │
     ▼
┌──────────────────────┐
│ Chunk 1:       A B C D E│
└──────────────────────┘
              ┌──────────────────────┐
              │ Chunk 2:       D E F G H │
              └──────────────────────┘
                              ┌──────────────────────┐
                              │ Chunk 3:       G H I J K │
                              └──────────────────────┘

       Overlapping text preserves continuity
       between consecutive chunks.
```

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,       # ~150 words per chunk
    chunk_overlap=100,    # overlap keeps context at boundaries
    separators=["\n\n", "\n", ".", " "],  # tries paragraph → line → sentence → word
)

chunks = splitter.split_documents(pages)

print(f"Total chunks: {len(chunks)}  (from {len(pages)} pages)")


Total chunks: 37  (from 9 pages)


In [6]:
print("\n--- Example chunk ---\n")
print("First Chunck:\n")
print(chunks[0].page_content)


--- Example chunk ---

First Chunck:

Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [7]:
print("Second Chunck :\n")

print(chunks[1].page_content)

Second Chunck :

Telecom Technical Reference Guide  - Internal Use Only
1. Introduction to Mobile Networks
Mobile networks have evolved through several generations, each offering significant improvements in speed,
capacity, and capability.
2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to
around 50 kbps, sufficient only for text messaging and simple email.
3G (UMTS/HSPA) networks brought mobile broadband, enabling video calls, mobile internet browsing, and app


In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3182.25it/s]


Vector store ready. 37 vectors stored.


In [12]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

test_query = "What is VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

print(f"Query: {test_query}\n")
print(f"Retrieved {len(retrieved)} chunks:\n")
for i, doc in enumerate(retrieved, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()

Query: What is VoLTE and how does it improve call quality?

Retrieved 3 chunks:

--- Chunk 1 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- Chunk 2 ---
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt

--- Chunk 3 ---
prioritised over general data traffic. This prevents voice quality degradation during periods of network congestion.
Without QoS, voice packets would compete with video streaming and file downloads, causing jitter and packet
loss.
Fallback Behaviour: If a VoLTE call 

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama


#**Build the RAG Chain**

Now we wire everything together using LangChain's **LCEL (LangChain Expression Language)**:

```
question
   │
   ├──► retriever ──► join chunks into one string ──►  context
   │                                                        │
   └────────────────────────────────────────────►  prompt template
                                                            │
                                                           LLM
                                                            │
                                                       Answer (string)
```

The system prompt tells the LLM to *only* use the retrieved context — this prevents hallucination.

In [16]:
# --- Helper function in the chain: join retrieved chunks into a single context string ---
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

# --- System prompt
SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

# --- LLM via Groq API ---
#llm = ChatGroq(
#    model="openai/gpt-oss-20b" or "/qwen/qwen3-32b",
#    temperature=0,
#    reasoning_format="parsed",
#)

# --- Assemble the chain ---

llm = ChatOllama(
    model="qwen3:8b",
    temperature=0
)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled.")

RAG chain assembled.


---
**Ask a Single Question**
-

In [17]:
question = "How does international roaming work and what charges should I expect?"

print(f"Q: {question}\n")
print("A:", chain.invoke(question))

Q: How does international roaming work and what charges should I expect?

A: International roaming works by connecting your device to a partner network in the visited country when you're outside your home network's coverage. Here's how it works:  
1. **Authentication**: The visited network uses protocols like SS7 or Diameter to authenticate your device.  
2. **Validation**: Your home network checks your subscription and authorizes services.  
3. **Tunneling**: All data, voice, and SMS traffic is routed back to your home network for billing, which may introduce latency compared to local usage.  

**Charges and Zones**:  
- **Zone A** (EU, UK, Australia, New Zealand): Lowest roaming rates.  
- **Zone B** (USA, Canada, Japan, Singapore): Moderate rates.  
- **Zone C** (Rest of World): Highest per-MB and per-minute charges.  

**Recommendations**:  
- Purchase a roaming bundle before traveling to **Zone B or C** to avoid high charges.  
- Pre-bundle data usage at standard rates isn’t refun

---
**Interactive Q&A Loop**
-
```
Try these questions:

"What is the difference between 4G and 5G?"
"How do I fix a SIM card that is not being detected?"
"What is SIM swap fraud and how can I protect myself?"
"Explain VoLTE in simple terms."
"What happens during a billing dispute?"
```

**Type `quit` to exit.**


In [19]:
print("Telecom RAG Assistant — type 'quit' to exit\n")

while True:
    question = input("Your question: ").strip()
    if question.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    if not question:
        continue

    print("\nAnswer:")
    for chunk in chain.stream(question):
        print(chunk, end="", flush=True)
    print("\n")

Telecom RAG Assistant — type 'quit' to exit



Your question:  what is the difference between 4G and 5G?



Answer:
The key differences between 4G and 5G are as follows:  

1. **Speed**:  
   - **4G (LTE)**: Typical download speeds range from 20–150 Mbps, with LTE-Advanced reaching up to 300 Mbps.  
   - **5G**: Targets peak speeds of **10 Gbps**, significantly faster than 4G.  

2. **Latency**:  
   - **4G**: Latency is below **50 ms**.  
   - **5G**: Achieves latency under **1 ms**, enabling real-time applications.  

3. **Device Connectivity**:  
   - **4G**: Supports fewer devices per area.  
   - **5G**: Can connect up to **one million devices per square kilometre**, critical for IoT and dense networks.  

4. **Frequency Bands**:  
   - **4G**: Operates primarily on mid-range frequencies (e.g., 2.6 GHz).  
   - **5G**: Uses **sub-6 GHz** and **millimetre-wave (mmWave)** frequencies. Lower frequencies offer broader coverage, while higher frequencies enable faster speeds but shorter range.  

5. **Use Cases**:  
   - **4G**: Enabled smartphone adoption and mobile broadband.  
   - **5G**

Your question:  Explain VoLTE in simple terms.



Answer:
VoLTE (Voice over LTE) is a technology that lets you make voice calls using your 4G LTE data network instead of the older 2G/3G voice networks. Here’s what it means in simple terms:  

1. **HD Voice Quality**: Calls sound clearer and more natural, like a wider range of sounds (think of it as "wideband audio").  
2. **Faster Calls**: Your call connects almost instantly (under 2 seconds) instead of waiting 6-8 seconds like older networks.  
3. **No Call Drop When Using Data**: You can browse the internet or use apps while talking, unlike older networks where data and calls couldn’t happen at the same time.  
4. **Network Backup**: If VoLTE fails (e.g., the other person’s network doesn’t support it), your call automatically switches to a 3G network, but with lower quality and no data during the call.  

To use VoLTE, your phone, SIM card, and carrier must all support it.



Your question:  quit


Goodbye!


---
## Bonus — Inspect What Was Retrieved

This cell shows you exactly which chunks were retrieved for a given question, before the LLM sees them.  
Great for debugging retrieval quality.

In [20]:
debug_question = "What security measures protect against SIM swap fraud?"

docs = retriever.invoke(debug_question)
print(f"Question: {debug_question}")
print(f"Retrieved {len(docs)} chunks:\n")
for i, doc in enumerate(docs, 1):
    print(f"{'='*60}")
    print(f"Chunk {i} (page {doc.metadata.get('page', '?')})")
    print(f"{'='*60}")
    print(doc.page_content)
    print()

print("\nFinal Answer:")
print(chain.invoke(debug_question))

Question: What security measures protect against SIM swap fraud?
Retrieved 3 chunks:

Chunk 1 (page 8)
mitigate this with SS7 firewalls and anomaly detection systems, but the risk cannot be fully eliminated on legacy
protocols. 5G's use of HTTPS-based APIs (Service Based Architecture) substantially reduces this attack surface.
SIM Swap Fraud: Described in Section 5. Key mitigation: enforce strict in-person or multi-factor remote identity
verification before any SIM replacement. Flag accounts with recent SIM swaps for elevated fraud monitoring for
30 days.
International Revenue Share Fraud (IRSF): Fraudsters compromise a PBX or customer account and generate

Chunk 2 (page 5)
provide an additional layer of protection; after three incorrect PIN attempts the SIM is locked and requires a PUK
code to unlock.
SIM Swap Fraud: SIM swap attacks occur when a fraudster convinces a carrier to transfer a victim's number to a
new SIM. This allows the attacker to intercept SMS-based two-factor authent